# Radar Financeiro V6

Preparar a tabela inicial com uma linha por MCI avaliado.

Na implementação, usar o mínimo de variáveis intermediárias. As regras de elegibilidade entram no cálculo das flags. Tratar CPF, agência, conta e outros valores no passo em que forem usados. Acrescentar validações separadas apenas se houver uma necessidade concreta.


## 1. Conexão

> **Conexão protegida. É proibido alterar, substituir ou excluir as células de conexão.**


Use os gerenciadores carregados abaixo para acessar as fontes do Radar. O `gerenciador_spark_v2.ipynb` fornece utilitários para a sessão Spark já iniciada, e o `gerenciador_db2_spark_v2.ipynb` fornece o conector para consultas ao DB2.


In [1]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-todos-1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
48732,application_1794695782786_37113,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.
Sessão Spark inicializada.


In [2]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


Módulo gerenciador_spark carregado com sucesso.

Módulo gerenciador_db2_spark carregado com sucesso (ConectorDb2Spark).

## 2. Parâmetros da execução

Usar a data `HOJE` do ambiente Spark e a mesma quantidade de ciclos para a renda presumida e o período de análise.


In [ ]:
%%spark
from datetime import date

dt_exea = date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])
quantidade_ciclos = 1
conector_db2 = criar_conector_db2_spark()


## 3. Entrada do MCI

Primeira etapa da apresentação. A leitura de transações recentes forma o Público Base, fornece a última sincronização e identifica pessoa física. A classificação Agro complementa a base por MCI.


### Filtro inicial pela data de sincronização — Público Base

Em `DB2GFP.TRAN_RLZD_INST_PCT`, selecionar os MCIs com transações de estado `CD_EST_TRAN_INST = 0` incluídas na janela de sincronização. O início da janela é inclusivo (um mês antes de `HOJE`) e o fim é exclusivo (`HOJE`). O Público Base contém um `CD_CLI_REF` por MCI.


In [ ]:
%%spark
# Reutilizar a leitura na última sincronização e nas flags de elegibilidade.
transacoes_recentes = conector_db2.sql(f"""
    SELECT CD_CLI, TS_INCL_TRAN, CD_TIP_PSS, NR_CPF_CNPJ_TITR,
           NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
    FROM DB2GFP.TRAN_RLZD_INST_PCT
    WHERE CD_CLI IS NOT NULL
      AND CD_EST_TRAN_INST = 0
      AND TS_INCL_TRAN >= TIMESTAMP(DATE('{dt_exea}') - 1 MONTH)
      AND TS_INCL_TRAN < TIMESTAMP(DATE('{dt_exea}'))
""").persist()

publico_base = transacoes_recentes.selectExpr("CAST(CD_CLI AS INT) AS CD_CLI_REF").distinct()


### Adicionar a última sincronização por MCI

Na mesma leitura do Público Base, calcular `MAX(TS_INCL_TRAN)` por MCI e registrar em `TS_INCL_TRAN_REF`.


In [ ]:
%%spark
publico_base = (
    transacoes_recentes
    .selectExpr("CAST(CD_CLI AS INT) AS CD_CLI_REF", "TS_INCL_TRAN")
    .groupBy("CD_CLI_REF")
    .agg(F.max("TS_INCL_TRAN").alias("TS_INCL_TRAN_REF"))
)


### Identificar Agro por MCI

Consultar as classificações principais de Agro por MCI em `DB2SMI.SUB_NHMC_CLI` e excluir a categoria 6 de `DB2M1A.CLI`. Registrar `FLAG_AGRO` como `'true'` para os MCIs encontrados e `'false'` para os demais. A flag é usada na decisão final de participação.


In [ ]:
%%spark
agro_por_mci = conector_db2.sql("""
    SELECT DISTINCT
        CAST(agro.CD_CLI AS INTEGER) AS CD_CLI_REF,
        'true' AS FLAG_AGRO
    FROM DB2SMI.SUB_NHMC_CLI agro
    WHERE agro.CD_CLI IS NOT NULL
      AND agro.IN_NHMC_PPL_CLI = 'S'
      AND agro.NR_SEQL_NHMC IN (31, 59, 60, 61, 76, 88, 125, 165)
      AND NOT EXISTS (
          SELECT 1
          FROM DB2M1A.CLI cli
          WHERE cli.CD_CLI = agro.CD_CLI
            AND cli.CD_CTGR = 6
      )
""")

publico_base = (
    publico_base.drop("FLAG_AGRO")
    .join(agro_por_mci, "CD_CLI_REF", "left")
    .fillna("false", subset=["FLAG_AGRO"])
)


### Identificar pessoa física

Na leitura de transações recentes, marcar `FLAG_PESSOA_FISICA` como `'true'` quando houver pelo menos um registro do MCI com `CD_TIP_PSS = 1`. Os demais MCIs recebem `'false'` e permanecem na tabela inicial.


In [ ]:
%%spark
pessoas_fisicas = (
    transacoes_recentes
    .where("CD_TIP_PSS = 1")
    .selectExpr("CAST(CD_CLI AS INT) AS CD_CLI_REF", "'true' AS FLAG_PESSOA_FISICA")
    .distinct()
)

publico_base = (
    publico_base.drop("FLAG_PESSOA_FISICA")
    .join(pessoas_fisicas, "CD_CLI_REF", "left")
    .fillna("false", subset=["FLAG_PESSOA_FISICA"])
)


## 4. Identificação e contexto

Segunda etapa da apresentação. Cada regra enriquece o mesmo DataFrame `contexto`. Uma regra aprovada permite seguir. As flags das regras que o MCI ainda não alcançou ficam nulas.


### Identificar conta corrente BB única

Apurar, para cada MCI identificado como pessoa física, se há exatamente um par elegível de agência e conta que possa ser convertido para a consulta do ciclo. Registrar `FLAG_CONTA_CORRENTE_BB_UNICA`. Na mesma agregação, apurar os CPFs distintos nos registros de pessoa física, antes do filtro específico de conta BB.


In [ ]:
%%spark
def apurar_conta_e_cpf(transacoes):
    # Conta e CPF compartilham a agregação. O filtro BB vale apenas para a conta.
    agencia = F.trim(F.col("NR_AG_TITR").cast("string"))
    conta = F.trim(F.col("CD_CT_TITR").cast("string"))
    conta_bb = F.when(
        (F.col("NR_MCA_PCT_OPB") == 999999999)
        & (F.col("CD_PRD") == 6)
        & (agencia != "")
        & (conta != ""),
        F.struct(agencia.alias("agencia"), conta.alias("conta")),
    )
    return (
        transacoes.where("CD_TIP_PSS = 1")
        .withColumn("CD_CLI_REF", F.col("CD_CLI").cast("int"))
        .groupBy("CD_CLI_REF")
        .agg(
            F.countDistinct(conta_bb).alias("QT_CONTAS_BB"),
            F.min(conta_bb).alias("CONTA_BB"),
            F.countDistinct("NR_CPF_CNPJ_TITR").alias("QT_CPFS"),
            F.min("NR_CPF_CNPJ_TITR").alias("CPF"),
        )
        .withColumn("CD_UOR_CC", F.expr("""
            CASE WHEN QT_CONTAS_BB = 1 AND CONTA_BB.agencia RLIKE '^[0-9]+$'
                 THEN TRY_CAST(CONTA_BB.agencia AS INT)
            END
        """))
        .withColumn("NR_CC", F.expr("""
            CASE WHEN QT_CONTAS_BB = 1 AND CONTA_BB.conta RLIKE '^[0-9]+$'
                 THEN TRY_CAST(COALESCE(NULLIF(REGEXP_REPLACE(CONTA_BB.conta, '^0+', ''), ''), '0')
                               AS DECIMAL(11,0))
            END
        """))
        .drop("QT_CONTAS_BB", "CONTA_BB")
    )


contexto = (
    publico_base.join(apurar_conta_e_cpf(transacoes_recentes), "CD_CLI_REF", "left")
    .withColumn("FLAG_CONTA_CORRENTE_BB_UNICA", F.expr("""
        CASE WHEN FLAG_PESSOA_FISICA = 'true'
             THEN CAST(CD_UOR_CC IS NOT NULL AND NR_CC IS NOT NULL AS STRING)
        END
    """))
)


### Identificar CPF único

Para os MCIs aprovados na conta BB, usar a contagem já apurada para preencher `FLAG_CPF_UNICO`. Converter o CPF único para o formato numérico da consulta de renda.


In [ ]:
%%spark
contexto = (
    contexto
    .withColumn("FLAG_CPF_UNICO", F.expr("""
        CASE WHEN FLAG_CONTA_CORRENTE_BB_UNICA = 'true'
             THEN CAST(COALESCE(QT_CPFS = 1, FALSE) AS STRING)
        END
    """))
    .withColumn("CPF_UNICO", F.expr("""
        CASE WHEN FLAG_CPF_UNICO = 'true' THEN TRY_CAST(CPF AS DECIMAL(14,0)) END
    """))
    .drop("QT_CPFS", "CPF")
    .persist()
)

# Manter os resultados disponíveis para as consultas até montar a tabela.
etapas_em_cache = [contexto]


### Consultar renda presumida

Consultar `DB2DFE.REN_AVLD_PF` pelo CPF aprovado e usar o registro mais recente por `DT_INCL_REN_AVLD`. Registrar `VL_REN_PRES_REF` pela quantidade de ciclos e preencher `FLAG_RENDA_PRESUMIDA_DISPONIVEL`. Zero conta como renda disponível. O CPF é descartado após a consulta.


In [ ]:
%%spark
from functools import reduce
from itertools import islice


def consultar_em_lotes(base, chaves_spark, chaves_db2, consulta, esquema):
    chaves = base.select(*chaves_spark).dropna().distinct().toLocalIterator()
    resultados = []
    while lote := list(islice(chaves, 500)):
        if len(chaves_db2) == 1:
            valores = ", ".join(str(int(linha[0])) for linha in lote)
            filtro = f"{chaves_db2[0]} IN ({valores})"
        else:
            filtro = " OR ".join(
                "(" + " AND ".join(
                    f"{coluna} = {int(valor)}" for coluna, valor in zip(chaves_db2, linha)
                ) + ")"
                for linha in lote
            )
        resultados.append(conector_db2.sql(consulta.format(filtro=filtro)))

    if resultados:
        return reduce(lambda a, b: a.unionByName(b), resultados)
    return base.sparkSession.createDataFrame([], esquema)


renda = consultar_em_lotes(
    contexto.where("FLAG_CPF_UNICO = 'true'"),
    ["CPF_UNICO"], ["NR_CPF_BASE_SRF"],
    f"""
    SELECT
        CAST(NR_CPF_BASE_SRF AS DECIMAL(14,0)) AS CPF_UNICO,
        CAST(VL_REN * {quantidade_ciclos} AS DECIMAL(17,2)) AS VL_REN_PRES_REF
    FROM (
        SELECT NR_CPF_BASE_SRF, VL_REN,
               ROW_NUMBER() OVER (
                   PARTITION BY NR_CPF_BASE_SRF ORDER BY DT_INCL_REN_AVLD DESC NULLS LAST
               ) AS RN
        FROM DB2DFE.REN_AVLD_PF
        WHERE {{filtro}}
    ) R
    WHERE RN = 1
    """,
    "CPF_UNICO DECIMAL(14,0), VL_REN_PRES_REF DECIMAL(17,2)",
)
contexto = (
    contexto.join(renda, "CPF_UNICO", "left")
    .withColumn("FLAG_RENDA_PRESUMIDA_DISPONIVEL", F.expr("""
        CASE WHEN FLAG_CPF_UNICO = 'true'
             THEN CAST(VL_REN_PRES_REF IS NOT NULL AS STRING)
        END
    """))
    .drop("CPF_UNICO")
    .persist()
)
etapas_em_cache.append(contexto)


### Consultar macroperfil

Consultar `DB2D1D.DVS_GRDR_FNCO_PF` pelo MCI aprovado em renda. Usar a maior `DT_REF` até a data da execução para preencher `CD_MAC_PRFL_CLI_REF`. Os códigos 1, 2 e 3 aprovam `FLAG_MACROPERFIL_VALIDO`.


In [ ]:
%%spark
perfil = consultar_em_lotes(
    contexto.where("FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'"),
    ["CD_CLI_REF"], ["CD_CLI"],
    f"""
    SELECT CAST(CD_CLI AS INTEGER) AS CD_CLI_REF,
           CAST(CD_MAC_PRFL_CLI AS INTEGER) AS CD_MAC_PRFL_CLI_REF,
           QT_PERFIS
    FROM (
        SELECT CD_CLI, CD_MAC_PRFL_CLI,
               ROW_NUMBER() OVER (PARTITION BY CD_CLI ORDER BY DT_REF DESC) AS RN,
               COUNT(*) OVER (PARTITION BY CD_CLI, DT_REF) AS QT_PERFIS
        FROM DB2D1D.DVS_GRDR_FNCO_PF
        WHERE DT_REF <= DATE('{dt_exea.isoformat()}') AND ({{filtro}})
    ) P
    WHERE RN = 1
    """,
    "CD_CLI_REF INT, CD_MAC_PRFL_CLI_REF INT, QT_PERFIS BIGINT",
)
contexto = (
    contexto.join(perfil, "CD_CLI_REF", "left")
    .withColumn("CD_MAC_PRFL_CLI_REF", F.expr("""
        CASE WHEN QT_PERFIS > 1
             THEN CAST(raise_error('Mais de um macroperfil na maior DT_REF do MCI') AS INT)
             ELSE CD_MAC_PRFL_CLI_REF
        END
    """))
    .drop("QT_PERFIS")
    .withColumn("FLAG_MACROPERFIL_VALIDO", F.expr("""
        CASE WHEN FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'
             THEN CAST(COALESCE(CD_MAC_PRFL_CLI_REF IN (1, 2, 3), FALSE) AS STRING)
        END
    """))
    .persist()
)
etapas_em_cache.append(contexto)


### Consultar o dia do ciclo financeiro

Consultar `DB2GFP.CT_GRDR_FNCO` por agência e conta para os MCIs aprovados nas regras anteriores. Preencher `DD_INC_MM_CLC_BLC_REF` com o dia do registro mais recente por `TS_ULT_EXEA_PSQ`, com valor nulo quando o cadastro não informar o dia. Agência e conta são descartadas após a consulta.


In [ ]:
%%spark
ciclo = consultar_em_lotes(
    contexto.where("FLAG_MACROPERFIL_VALIDO = 'true'"),
    ["CD_UOR_CC", "NR_CC"], ["CD_UOR_CC", "NR_CC"],
    """
    SELECT CD_UOR_CC, NR_CC, CAST(DD_INC_MM_CLC_BLC AS SMALLINT) AS DD_INC_MM_CLC_BLC_REF
    FROM (
        SELECT CD_UOR_CC, NR_CC, DD_INC_MM_CLC_BLC,
               ROW_NUMBER() OVER (
                   PARTITION BY CD_UOR_CC, NR_CC ORDER BY TS_ULT_EXEA_PSQ DESC NULLS LAST
               ) AS RN
        FROM DB2GFP.CT_GRDR_FNCO
        WHERE {filtro}
    ) C
    WHERE RN = 1
    """,
    "CD_UOR_CC INT, NR_CC DECIMAL(11,0), DD_INC_MM_CLC_BLC_REF SMALLINT",
)

contexto = (
    contexto.join(ciclo, ["CD_UOR_CC", "NR_CC"], "left")
    .withColumn("DD_INC_MM_CLC_BLC_REF", F.when(
        F.col("FLAG_MACROPERFIL_VALIDO") == "true", F.col("DD_INC_MM_CLC_BLC_REF")
    ))
    .drop("CD_UOR_CC", "NR_CC")
)


## 5. Período de análise

Para os MCIs que atendem às seis condições, localizar o ciclo em andamento na data da última sincronização. O período fechado termina na véspera desse ciclo e recua pela quantidade de ciclos definida na execução. Usar o dia 1 quando faltar o dia cadastrado e ajustar o início de cada ciclo ao último dia dos meses mais curtos.


In [ ]:
%%spark
condicoes_participacao = F.expr("""
    FLAG_AGRO = 'true'
    AND FLAG_PESSOA_FISICA = 'true'
    AND FLAG_CONTA_CORRENTE_BB_UNICA = 'true'
    AND FLAG_CPF_UNICO = 'true'
    AND FLAG_RENDA_PRESUMIDA_DISPONIVEL = 'true'
    AND FLAG_MACROPERFIL_VALIDO = 'true'
""")


def inicio_ciclo_no_mes(mes, dia):
    return F.date_add(mes, F.least(dia, F.dayofmonth(F.last_day(mes))) - 1)


data_sincronizacao = F.to_date("TS_INCL_TRAN_REF")
mes_sincronizacao = F.trunc(data_sincronizacao, "month")
dia_ciclo = F.coalesce(F.col("DD_INC_MM_CLC_BLC_REF").cast("int"), F.lit(1))
inicio_no_mes = inicio_ciclo_no_mes(mes_sincronizacao, dia_ciclo)
inicio_ciclo_aberto = F.when(data_sincronizacao >= inicio_no_mes, inicio_no_mes).otherwise(
    inicio_ciclo_no_mes(F.add_months(mes_sincronizacao, -1), dia_ciclo)
)
mes_inicio_periodo = F.add_months(F.trunc(inicio_ciclo_aberto, "month"), -quantidade_ciclos)

contexto = (
    contexto
    .withColumn("DT_INI_REF", F.when(
        condicoes_participacao, inicio_ciclo_no_mes(mes_inicio_periodo, dia_ciclo)
    ))
    .withColumn("DT_FIM_REF", F.when(
        condicoes_participacao, F.date_sub(inicio_ciclo_aberto, 1)
    ))
)


## 6. Montagem da tabela inicial

Depois do cálculo do período, registrar `FLAG_PARTICIPA_RADAR` como `'true'` para quem atende às seis condições e `'false'` nos demais casos. Entregar `tabela_inicial` com uma linha por MCI e os 15 campos na ordem do DDL.


In [ ]:
%%spark
try:
    tabela_inicial = (
        contexto
        .withColumn("DT_EXEA", F.lit(dt_exea))
        .withColumn("FLAG_PARTICIPA_RADAR", F.coalesce(condicoes_participacao, F.lit(False)).cast("string"))
        .select(
            "DT_EXEA", "CD_CLI_REF", "TS_INCL_TRAN_REF", "FLAG_AGRO",
            "FLAG_PESSOA_FISICA", "FLAG_CONTA_CORRENTE_BB_UNICA", "DD_INC_MM_CLC_BLC_REF",
            "FLAG_CPF_UNICO", "FLAG_RENDA_PRESUMIDA_DISPONIVEL", "VL_REN_PRES_REF",
            "FLAG_MACROPERFIL_VALIDO", "CD_MAC_PRFL_CLI_REF",
            "DT_INI_REF", "DT_FIM_REF", "FLAG_PARTICIPA_RADAR",
        )
        .localCheckpoint(eager=True)
    )
finally:
    for etapa in etapas_em_cache:
        etapa.unpersist()


## 7. Gravação da tabela inicial

Gravar o resultado completo em `SBX_A2A13E2.ANL_RADAR_FNC_CLI`, criada pelo notebook de DDL. Esta célula substitui os dados existentes pelo resultado da execução atual.


In [ ]:
%%spark
tabela_inicial.write.insertInto("SBX_A2A13E2.ANL_RADAR_FNC_CLI", overwrite=True)
